# Semiconductor demand exploration

Observed facts and scenario assumptions answer different questions. This notebook reads the reproducible CSV exports. Run src/analysis.py first. Source coverage is a small curated snapshot, not a live market dataset.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'data/processed').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/processed').exists(), 'Run from project root or notebooks directory'
def read(name):
    return pd.read_csv(ROOT / 'data/processed' / (name + '.csv'))
revenue = read('company_revenue')
market = read('market')
scenarios = read('scenarios')
sources = read('sources')
print(revenue.shape, market.shape, scenarios.shape)


## Audit completeness and provenance

Missing YoY growth for a company’s first included year is expected. Missing publication dates mean the exact date was not established; access dates are stored separately.


In [ ]:
print(revenue.isna().sum())
assert revenue['source_id'].isin(sources['source_id']).all()
assert not revenue.duplicated(['company_id', 'fiscal_year']).any()
assert revenue.groupby('company_id')['currency'].nunique().max() == 1
print(sources[['source_id', 'title', 'published_date']].to_string(index=False))


## Historical revenue patterns

Compare growth within each company, retaining different fiscal periods and currencies. No company revenue total or chip-market share is computed.


In [ ]:
latest = revenue.sort_values('fiscal_year').groupby('company_id').tail(1)
print(latest[['company', 'fiscal_year', 'period_end', 'currency', 'yoy_growth']].to_string(index=False))
latest.set_index('company')['yoy_growth'].sort_values().plot.barh(title='Latest included annual revenue change')
plt.xlabel('Fractional change versus previous fiscal year')
plt.tight_layout()
plt.show()


## Market vintage selection

The original and revised 2024 values remain visible. Select one before plotting or aggregating. Differences across release vintages can affect computed growth rates.


In [ ]:
selected = market.loc[market['selected'] == True].sort_values('year')
assert not selected['year'].duplicated().any()
print(market.to_string(index=False))
selected.plot(x='year', y='value_usd_billions', marker='o', title='Selected reported market history')
plt.ylabel('USD billions')
plt.show()


## Assumption sensitivity through 2036

All themes start at 100. Ranking the resulting index ranks our assumed compounding rates. It does not establish future dollar demand. Power electronics and design/packaging have identical base rates in this example and overlap.


In [ ]:
endpoints = scenarios.loc[scenarios['year'] == 2036]
comparison = endpoints.pivot(index='theme', columns='scenario', values='index_value')
print(comparison.round(1).to_string())
comparison.plot.barh(title='Illustrative 2036 indices: 2026 = 100')
plt.xlabel('Index; analyst assumptions')
plt.tight_layout()
plt.show()


## Interpretation

Historical growth differs across the value chain. The current evidence supports investigating AI infrastructure and its dependencies; it cannot establish a ten-year winner. Next collect compatible market-size, unit shipment, price and adoption series. See reports/findings.md and docs/methodology.md.
